# 02. Time-series validation

랜덤 분할로 미래 시즌의 분포를 과거 학습에 섞지 않고, 2022~2024년을 순서대로 검증합니다.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))
from src.metrics import brier_score, brier_skill_score, reference_brier_score
from src.validation import expanding_season_splits

DATA_PATH = PROJECT_ROOT / 'data' / 'train.csv'
train = pd.read_csv(DATA_PATH) if DATA_PATH.exists() else None

In [ ]:
if train is not None:
    splits = expanding_season_splits(train)
    for split in splits:
        print(split.train_seasons, '->', split.validation_season,
              len(split.train_index), len(split.validation_index))
else:
    print('Place a private train.csv in data/ to build the season splits.')

## Probability metric

`Brier Score = mean((p_i - y_i)^2)`이며 낮을수록 좋습니다. 평균 제구율을 `r`이라고 하면 상수 예측의 기준 점수는 `r * (1 - r)`입니다.

In [ ]:
# Small metric smoke test; these values are illustrative and are not competition scores.
y_example = np.array([0, 1, 1, 0])
p_example = np.array([0.2, 0.7, 0.6, 0.3])
print('Brier:', brier_score(y_example, p_example))
print('Reference:', reference_brier_score(y_example))
print('BSS:', brier_skill_score(y_example, p_example))

## Selection rule

모델과 특성은 한 fold의 최고점보다 여러 검증 시즌의 평균, 최악 시즌의 변화, 확률 보정 상태를 함께 보고 선택합니다. 모든 집계값과 보정 계수는 각 fold의 학습 기간에서만 추정합니다.